In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, Flatten, MaxPool2D, GlobalAveragePooling2D, Dropout, Rescaling
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

## 1) The Dataset
- We are building a binary classifier for fire detection.
- Class 1: Fire
- Class 0: Non_Fire (No fire)
- The local dataset is located in the `Fire_dataset` directory, consisting of 500 images per class.

In [ ]:
from PIL import Image

def create_dataset(img_folder):
    img_data_array=[]
    class_name=[]
   
    # Ensure sorted subdirectories to get consistent label mapping
    for dir1 in sorted(os.listdir(img_folder)):
        dir_path = os.path.join(img_folder, dir1)
        if not os.path.isdir(dir_path):
            continue
        for file in os.listdir(dir_path):
            image_path = os.path.join(dir_path, file)
            try:
                img = Image.open(image_path).convert('RGB')
                img = img.resize((256, 256))
                image = np.array(img).astype('float32')
                img_data_array.append(image)
                # Map class folder names ('0' vs '1', or 'Non_Fire' vs 'Fire') to integers (0 and 1)
                if dir1 == '1' or dir1.lower() == 'fire':
                    class_name.append(1)
                else:
                    class_name.append(0)
            except Exception as e:
                print(f"Error loading {image_path}: {e}")
    return np.array(img_data_array), np.array(class_name)

In [ ]:
# Load dataset
X, y = create_dataset("Fire_dataset")
print("Dataset loaded successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts (0=Non_Fire, 1=Fire):", np.bincount(y))

## 2) Pre-processing & Dataset Splits
We split our dataset into:
- 80% Training
- 10% Validation
- 10% Testing
We stratify by y to ensure balanced class distributions in each split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_pre, y_train, y_pre = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_pre, y_pre, test_size=0.5, random_state=0, stratify=y_pre)

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

## 3) MobileNetV2 Transfer Learning Model
We use a pre-trained **MobileNetV2** base model, frozen to preserve the rich feature extractor trained on ImageNet.
We append a lightweight classifier head on top (Global Average Pooling, Dropout, Dense) to avoid overfitting on our small dataset.
A built-in Rescaling layer is added to normalize images automatically.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Early Stopping to prevent overfitting
monitor_val_acc = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

# Data augmentation generator
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

def compile_model(model, lr=0.001):
    # Using modern Keras 'learning_rate' argument instead of deprecated 'lr'
    opt = Adam(learning_rate=lr)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return model

In [ ]:
# Load pre-trained MobileNetV2
transfer_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
transfer_model.trainable = False

# Build model using functional API
inputs = keras.Input(shape=(256, 256, 3))

# Internal Rescaling layer for preprocessing (scales raw inputs to [-1, 1] automatically)
x = Rescaling(scale=1./127.5, offset=-1.0)(inputs)
x = transfer_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, output)
model.summary()

In [ ]:
# Compile the model
model = compile_model(model, lr=0.001)

# Train the model
print("Training the MobileNetV2 Transfer Learning model...")
hist = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=15,
    callbacks=[monitor_val_acc],
    validation_data=(X_val, y_val)
)

In [ ]:
# Evaluate the model on test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.2%}")

## 4) Save the Trained Model
We save the model to `fire_detection_model.h5` so that it can be loaded directly by our webcam application `app.py`.

In [ ]:
model.save('fire_detection_model.h5')
print("Model saved successfully as 'fire_detection_model.h5'!")

## 5) Training History Plots
We plot training and validation loss and accuracy curves to evaluate performance.

In [ ]:
loss = hist.history['loss']
val_loss = hist.history['val_loss']
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
epochs_range = range(1, len(loss) + 1)

plt.figure(figsize=(14, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Val Loss', marker='x')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.title('Training & Validation Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Train Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Val Accuracy', marker='x')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.title('Training & Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## 6) Performance Metrics
Detailed metrics on test data including the Confusion Matrix, F1 Score, and Recall.

In [ ]:
import sklearn.metrics as metrics

# Get probability predictions
y_pred = model.predict(X_test)
# Convert probabilities to binary predictions
y_pred_binary = (y_pred > 0.5).astype(int).squeeze()

# Plot confusion matrix
cm = metrics.confusion_matrix(y_test, y_pred_binary)
disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non_Fire", "Fire"])
disp.plot(cmap=plt.cm.Oranges)
plt.title('Confusion Matrix')
plt.show()

# Compute metrics
f1 = metrics.f1_score(y_test, y_pred_binary)
recall = metrics.recall_score(y_test, y_pred_binary)
precision = metrics.precision_score(y_test, y_pred_binary)

print(f"Test Precision: {precision:.4f}")
print(f"Test Recall (Sensitivity): {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

## 7) Error Analysis
Visualizing test samples that were misclassified by the model.

In [ ]:
incorrect_indices = np.nonzero(y_pred_binary != y_test)[0]
print(f"Total incorrect predictions: {len(incorrect_indices)}")

if len(incorrect_indices) > 0:
    plt.figure(figsize=(10, 10))
    # Display up to 9 incorrect samples
    for i, incorrect in enumerate(incorrect_indices[:9]):
        plt.subplot(3, 3, i + 1)
        plt.imshow(X_test[incorrect].astype('uint8'))
        plt.axis('off')
        plt.title(f"Pred: {'Fire' if y_pred_binary[incorrect] == 1 else 'Non_Fire'}\nTrue: {'Fire' if y_test[incorrect] == 1 else 'Non_Fire'}")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified samples found!")